<a href="https://colab.research.google.com/github/JulioLaz/RAG_alura_prueba/blob/main/RAG_OpenRouter_LangGraph_etica.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 RAG Agent con OpenRouter + LangGraph

**Arquitectura:**
```
Documento (PDF/DOCX) → Fragmentar → Embeddings (HuggingFace) → FAISS
                                                                    ↓
Pregunta → [Agente LangGraph] → ¿RAG o LLM directo? → Respuesta
```

**Stack:**
- 🔗 LangChain + LangGraph
- 🧠 LLM: OpenRouter (cascada de modelos `:free`)
- 📐 Embeddings: HuggingFace `sentence-transformers` (sin API key)
- 📦 Vector Store: FAISS

**Requisito único:** API Key de [openrouter.ai](https://openrouter.ai) → guardarla en **Colab Secrets** como `OPENROUTER_KEY`

---
## PASO 1 — Instalación de dependencias

In [ ]:
%%capture
!pip install langchain langchain-community langchain-text-splitters
!pip install langgraph
!pip install faiss-cpu
!pip install sentence-transformers
!pip install pypdf docx2txt
!pip install requests

print('✅ Dependencias instaladas correctamente')

---
## PASO 2 — Configuración: API Key y parámetros

In [ ]:
'''
RAG Agent con OpenRouter + LangGraph
Consulta documentos (PDF/DOCX) via RAG o responde directamente con LLM.
LLM: OpenRouter con cascada automática de modelos free.
Embeddings: HuggingFace sentence-transformers (sin API key).
'''

# ═══════════════════════════════════════════════════════
# RAG_OpenRouter_LangGraph.ipynb
# Versión: 1.1
# Cambios:
#   v1.0 - Pipeline completo RAG + LangGraph + OpenRouter cascada
#   v1.1 - Fix modelos 404, agrego openrouter/free como router automático
#          Fix nodo_clasificador con llamadas extra, soporte DOCX
# ═══════════════════════════════════════════════════════

import os
import requests
from datetime import datetime
from google.colab import userdata

# ── API Key desde Colab Secrets ──────────────────────────
OPENROUTER_KEY    = userdata.get('OPENROUTER_KEY')
OPENROUTER_ENDPOINT = 'https://openrouter.ai/api/v1/chat/completions'
MAX_TOKENS        = 512

# ── Lista de modelos free (marzo 2026) ───────────────────
MODELOS_PREFERIDOS = [
    'openrouter/free',                              # router automático OpenRouter
    'google/gemini-2.0-flash-exp:free',
    'meta-llama/llama-3.3-70b-instruct:free',
    'mistralai/mistral-small-3.1-24b-instruct:free',
    'google/gemma-3-12b-it:free',
    'google/gemma-3-4b-it:free',
    'meta-llama/llama-3.2-3b-instruct:free',
    'allenai/olmo-3.1-32b-think:free',
]

ERRORES_REINTENTABLES = {429, 503}

print(f'✅ Configuración cargada | {datetime.now().strftime("%H:%M:%S")}')
print(f'   🔑 API Key: {"OK" if OPENROUTER_KEY else "❌ NO ENCONTRADA"}')
print(f'   🤖 Modelos en cascada: {len(MODELOS_PREFERIDOS)}')

---
## PASO 3 — LLM Custom: OpenRouter con cascada automática

In [ ]:
from langchain_core.language_models.llms import LLM
from typing import Optional, List
import time


class OpenRouterCascadaLLM(LLM):
    """
    LLM custom para LangChain que usa OpenRouter.
    Si un modelo devuelve 429/503, prueba el siguiente de la lista.
    Los 404 se saltan directamente (modelo inexistente).
    """

    @property
    def _llm_type(self) -> str:
        return 'openrouter_cascada'

    def _consultar_modelo(self, modelo: str, prompt: str) -> tuple:
        """Intenta un modelo. Retorna (texto | None, status_code)."""
        headers = {
            'Authorization': f'Bearer {OPENROUTER_KEY}',
            'Content-Type': 'application/json',
        }
        payload = {
            'model': modelo,
            'max_tokens': MAX_TOKENS,
            'messages': [{'role': 'user', 'content': prompt}]
        }
        try:
            resp = requests.post(
                OPENROUTER_ENDPOINT, headers=headers,
                json=payload, timeout=30
            )
            if resp.status_code == 200:
                texto = resp.json()['choices'][0]['message']['content']
                return texto, 200
            return None, resp.status_code
        except requests.exceptions.Timeout:
            return None, 408

    def _call(self, prompt: str, stop: Optional[List[str]] = None) -> str:
        """Prueba modelos en cascada hasta obtener respuesta."""
        print(f'\n⏳ Consultando OpenRouter...')
        for modelo in MODELOS_PREFERIDOS:
            print(f'   🔄 {modelo}...', end=' ', flush=True)
            texto, status = self._consultar_modelo(modelo, prompt)
            if status == 200:
                print('✅')
                return texto
            elif status in ERRORES_REINTENTABLES:
                print(f'⚠️  {status} — siguiente')
                time.sleep(3)
            elif status == 404:
                print(f'⛔ 404 modelo no existe — siguiente')
            elif status == 408:
                print('⏱️  Timeout — siguiente')
            else:
                print(f'❌ Error {status} — siguiente')
        raise RuntimeError('❌ Ningún modelo respondió correctamente.')


llm = OpenRouterCascadaLLM()
print('✅ LLM OpenRouter con cascada listo')

---
## PASO 4 — Carga y fragmentación de documentos (PDF y DOCX)

In [ ]:
from google.colab import files
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# ── Subir archivos ───────────────────────────────────────
print('📂 Seleccioná uno o más archivos PDF o DOCX...')
os.makedirs('docs', exist_ok=True)
uploader = files.upload()

for archivo in uploader.keys():
    os.rename(archivo, f'docs/{archivo}')

# ── Cargar documentos ────────────────────────────────────
documentos = []
for archivo in os.listdir('docs'):
    ruta = f'docs/{archivo}'
    if archivo.endswith('.pdf'):
        loader = PyPDFLoader(ruta)
    elif archivo.endswith('.docx'):
        loader = Docx2txtLoader(ruta)
    else:
        print(f'⚠️  Formato no soportado: {archivo}')
        continue
    documentos.extend(loader.load())

print(f'\n📄 Documentos cargados: {len(documentos)} páginas/secciones')

# ── Fragmentar ───────────────────────────────────────────
divisor = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=['\n\n', '\n', '. ', ' ', '']
)
fragmentos = divisor.split_documents(documentos)
print(f'✂️  Fragmentos generados: {len(fragmentos)}')

---
## PASO 5 — Embeddings HuggingFace + índice FAISS

> ⏳ Este paso tarda ~1-2 minutos la primera vez (descarga modelo ~90MB)

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

print('🔄 Cargando modelo de embeddings HuggingFace...')
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
)
print('✅ Modelo de embeddings listo')

print('\n🔄 Creando índice FAISS...')
inicio = datetime.now()
vectorstore = FAISS.from_documents(fragmentos, embeddings)
retriever   = vectorstore.as_retriever(search_kwargs={'k': 4})
elapsed     = (datetime.now() - inicio).total_seconds()

print(f'✅ Índice FAISS creado en {elapsed:.1f}s')
print(f'   📊 Vectores indexados: {vectorstore.index.ntotal}')

---
## PASO 6 — Agente LangGraph: clasifica y responde

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END


# ── Estado del agente ────────────────────────────────────
class AgentState(TypedDict):
    pregunta: str
    fuente:   str    # 'RAG' o 'DIRECTO'
    contexto: str
    respuesta: str


# ── Nodo 1: Clasificador ─────────────────────────────────
def nodo_clasificador(state: AgentState) -> AgentState:
    """Decide si responder con RAG (documentos) o directo (LLM)."""
    pregunta = state['pregunta']
    prompt = f"""Eres un clasificador. Dada la pregunta, decide si se debe
responder buscando en los DOCUMENTOS cargados o con conocimiento general.

Reglas:
- Responde SOLO 'RAG' si la pregunta se refiere al contenido de los documentos.
- Responde SOLO 'DIRECTO' si la pregunta es general o de conocimiento amplio.

Pregunta: {pregunta}

Responde SOLO con una palabra (RAG o DIRECTO):"""

    decision = llm.invoke(prompt).strip().upper()
    fuente = 'RAG' if 'RAG' in decision else 'DIRECTO'
    print(f'\n🧭 Clasificación: {fuente}')
    return {**state, 'fuente': fuente}


# ── Nodo 2: Búsqueda en documentos (RAG) ─────────────────
def nodo_rag(state: AgentState) -> AgentState:
    """Recupera contexto relevante del vectorstore y responde."""
    pregunta = state['pregunta']
    docs     = retriever.invoke(pregunta)
    contexto = '\n\n---\n\n'.join(doc.page_content for doc in docs)

    prompt = f"""Eres un asistente experto. Responde ÚNICAMENTE basándote
en el contexto provisto. Si la información no está, dilo claramente.

Contexto:
{contexto}

Pregunta: {pregunta}

Respuesta:"""

    respuesta = llm.invoke(prompt)
    return {**state, 'contexto': contexto, 'respuesta': respuesta}


# ── Nodo 3: Respuesta directa (sin docs) ─────────────────
def nodo_directo(state: AgentState) -> AgentState:
    """Responde con conocimiento general del LLM."""
    pregunta = state['pregunta']
    prompt = f"""Responde la siguiente pregunta de forma clara y concisa.

Pregunta: {pregunta}

Respuesta:"""

    respuesta = llm.invoke(prompt)
    return {**state, 'contexto': '(sin documentos)', 'respuesta': respuesta}


# ── Enrutamiento ──────────────────────────────────────────
def decidir_fuente(state: AgentState) -> str:
    return 'RAG_elegido' if state['fuente'] == 'RAG' else 'Directo_elegido'


# ── Construir grafo ───────────────────────────────────────
grafo = StateGraph(AgentState)
grafo.add_node('clasificador', nodo_clasificador)
grafo.add_node('rag',          nodo_rag)
grafo.add_node('directo',      nodo_directo)
grafo.add_edge(START, 'clasificador')
grafo.add_conditional_edges(
    'clasificador',
    decidir_fuente,
    {'RAG_elegido': 'rag', 'Directo_elegido': 'directo'}
)
grafo.add_edge('rag',     END)
grafo.add_edge('directo', END)

agente = grafo.compile()
print('✅ Agente LangGraph compilado')
print('   START → [Clasificador] → RAG o Directo → END')

---
## PASO 7 — (Opcional) Visualizar el grafo

In [ ]:
from IPython.display import Image, display

try:
    imagen = agente.get_graph().draw_mermaid_png()
    display(Image(imagen))
except Exception as e:
    print(f'ℹ️  No se pudo renderizar el grafo: {e}')

---
## PASO 8 — Función principal y pruebas

In [ ]:
def ejecutar_agente(pregunta: str) -> str:
    """Ejecuta el pipeline RAG completo para una pregunta."""
    inicio = datetime.now()
    print('=' * 60)
    print(f'❓ Pregunta: {pregunta}')
    print('─' * 60)

    resultado = agente.invoke({
        'pregunta': pregunta,
        'fuente':   '',
        'contexto': '',
        'respuesta': ''
    })

    elapsed = (datetime.now() - inicio).total_seconds()
    print('─' * 60)
    print(f'📌 Fuente:  {resultado["fuente"]}')
    print(f'⏱️  Tiempo:  {elapsed:.1f}s')
    print(f'\n💬 Respuesta:\n{resultado["respuesta"]}')
    print('=' * 60)
    return resultado['respuesta']

In [ ]:
# ── Prueba 1: sobre el documento (debería usar RAG) ──────
ejecutar_agente('¿De qué trata el documento cargado?')

In [ ]:
# ── Prueba 2: conocimiento general (debería ir DIRECTO) ──
ejecutar_agente('¿Cuántos mundiales de fútbol tiene Brasil?')

In [ ]:
# ── Prueba libre ─────────────────────────────────────────
mi_pregunta = input('❓ Escribí tu pregunta: ').strip()
if mi_pregunta:
    ejecutar_agente(mi_pregunta)